<a href="https://colab.research.google.com/github/rafaelcasemiro01/rede_neural/blob/main/1_Treinamento_Pima_Indians.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Pima Indians Diabetes — Treinamento do Modelo
> **Disciplina:** Redes Neurais  |  **Plataforma:** Google Colab  |  **Autor:** Rafael Casemiro

Este notebook realiza o **treinamento completo** de uma Rede Neural (MLP) no dataset
Pima Indians Diabetes e **salva o modelo treinado** em disco para uso posterior.

---
## Fluxo do Notebook
```
1. Instalar dependências
2. Carregar o dataset Pima Indians
3. Análise exploratória (EDA)
4. Pré-processamento dos dados
5. Treinar a Rede Neural (MLP)
6. Avaliar o modelo
7. Salvar modelo + scaler em disco
```

## 📦 Célula 1 — Instalação das Dependências
O Colab já possui `sklearn`, `pandas` e `numpy`. Apenas garantimos as versões corretas.

In [ ]:
# Verificar versões instaladas
import sklearn, pandas, numpy, joblib
print(f'scikit-learn : {sklearn.__version__}')
print(f'pandas       : {pandas.__version__}')
print(f'numpy        : {numpy.__version__}')
print('✅ Todas as dependências disponíveis!')

scikit-learn : 1.6.1
pandas       : 2.2.2
numpy        : 2.0.2
✅ Todas as dependências disponíveis!


## 📥 Célula 2 — Carregar o Dataset
O **Pima Indians Diabetes Dataset** contém 768 registros de mulheres indígenas Pima
com 8 características clínicas e 1 rótulo (0=sem diabetes, 1=com diabetes).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Colunas do dataset
COLUNAS = [
    'Gestacoes',          # número de gestações
    'Glicose',            # concentração de glicose (OGTT)
    'PressaoArterial',    # pressão diastólica (mmHg)
    'EspessuraPele',      # espessura da dobra cutânea (mm)
    'Insulina',           # insulina sérica 2h (mu U/ml)
    'IMC',                # índice de massa corporal
    'LinhagemuDiabetes',  # função de pedigree de diabetes
    'Idade',              # idade (anos)
    'Diabetes'            # 0 = sem diabetes | 1 = com diabetes
]

# URL do dataset (repositório público)
URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'

df = pd.read_csv(URL, names=COLUNAS)

print(f'✅ Dataset carregado: {df.shape[0]} pacientes, {df.shape[1]-1} características')
print(f'\nDistribuição da classe alvo:')
print(f'  Sem diabetes (0): {(df.Diabetes==0).sum()} pacientes ({(df.Diabetes==0).mean()*100:.1f}%)')
print(f'  Com diabetes (1): {(df.Diabetes==1).sum()} pacientes ({(df.Diabetes==1).mean()*100:.1f}%)')
print(f'\nPrimeiras 5 linhas:')
df.head()

✅ Dataset carregado: 768 pacientes, 8 características

Distribuição da classe alvo:
  Sem diabetes (0): 500 pacientes (65.1%)
  Com diabetes (1): 268 pacientes (34.9%)

Primeiras 5 linhas:


,Gestacoes,Glicose,PressaoArterial,EspessuraPele,Insulina,IMC,LinhagemuDiabetes,Idade,Diabetes
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 🔍 Célula 3 — Análise Exploratória (EDA)
Verificamos estatísticas e identificamos valores zerados que são clinicamente impossíveis.

In [ ]:
print('Estatísticas descritivas:')
print(df.describe().round(2).to_string())

# Zeros impossíveis (glicose=0, pressão=0, etc. são biologicamente inválidos)
cols_sem_zero = ['Glicose', 'PressaoArterial', 'EspessuraPele', 'Insulina', 'IMC']
print('\nContagem de zeros suspeitos (biologicamente impossíveis):')
for col in cols_sem_zero:
    n = (df[col] == 0).sum()
    pct = n / len(df) * 100
    print(f'  {col:<22}: {n:>3} zeros ({pct:.1f}%)')

Estatísticas descritivas:
       Gestacoes  Glicose  PressaoArterial  EspessuraPele  Insulina     IMC  LinhagemuDiabetes   Idade  Diabetes
count     768.00   768.00           768.00         768.00    768.00  768.00             768.00  768.00    768.00
mean        3.85   120.89            69.11          20.54     79.80   31.99               0.47   33.24      0.35
std         3.37    31.97            19.36          15.95    115.24    7.88               0.33   11.76      0.48
min         0.00     0.00             0.00           0.00      0.00    0.00               0.08   21.00      0.00
25%         1.00    99.00            62.00           0.00      0.00   27.30               0.24   24.00      0.00
50%         3.00   117.00            72.00          23.00     30.50   32.00               0.37   29.00      0.00
75%         6.00   140.25            80.00          32.00    127.25   36.60               0.63   41.00      1.00
max        17.00   199.00           122.00          99.00    846.00   

## ⚙️ Célula 4 — Pré-processamento
1. **Substituição de zeros** pela mediana da coluna (tratamento de dados faltantes)
2. **Divisão** treino/teste (80% / 20%)
3. **Normalização** com StandardScaler (média=0, desvio=1)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- 1. Tratar zeros impossíveis ---
df_clean = df.copy()
cols_sem_zero = ['Glicose', 'PressaoArterial', 'EspessuraPele', 'Insulina', 'IMC']

for col in cols_sem_zero:
    mediana = df_clean[col][df_clean[col] > 0].median()
    df_clean[col] = df_clean[col].replace(0, mediana)
    print(f'  {col:<22}: zeros → {mediana:.2f} (mediana)')

# --- 2. Separar features e alvo ---
X = df_clean.drop('Diabetes', axis=1).values
y = df_clean['Diabetes'].values
feature_names = df_clean.drop('Diabetes', axis=1).columns.tolist()

# --- 3. Dividir treino / teste ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # garante proporção igual em treino e teste
)

# --- 4. Normalizar (scaler aprende só no treino!) ---
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform no treino
X_test_sc  = scaler.transform(X_test)        # apenas transform no teste

print(f'\n✅ Pré-processamento concluído!')
print(f'   Treino : {X_train_sc.shape[0]} amostras')
print(f'   Teste  : {X_test_sc.shape[0]} amostras')
print(f'   Features: {X_train_sc.shape[1]}')

  Glicose               : zeros → 117.00 (mediana)
  PressaoArterial       : zeros → 72.00 (mediana)
  EspessuraPele         : zeros → 29.00 (mediana)
  Insulina              : zeros → 125.00 (mediana)
  IMC                   : zeros → 32.30 (mediana)

✅ Pré-processamento concluído!
   Treino : 614 amostras
   Teste  : 154 amostras
   Features: 8


## 🏋️ Célula 5 — Treinar a Rede Neural (MLP)
Usamos um **Perceptron Multicamadas (MLP)** com 3 camadas ocultas.
Arquitetura: `8 entradas → 64 → 32 → 16 → 1 saída`

In [ ]:
from sklearn.neural_network import MLPClassifier
import time

# Definir arquitetura da rede
modelo = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),  # 3 camadas ocultas
    activation='relu',                # função de ativação ReLU
    solver='adam',                    # otimizador Adam
    learning_rate_init=0.001,         # taxa de aprendizado inicial
    max_iter=1000,                    # máximo de épocas
    early_stopping=True,              # para se não melhorar
    validation_fraction=0.1,          # 10% do treino para validação
    n_iter_no_change=20,              # paciência: 20 épocas sem melhora
    random_state=42                   # reprodutibilidade
)

print('Arquitetura da Rede Neural:')
print('  Entrada  : 8 features')
print('  Camada 1 : 64 neurônios (ReLU)')
print('  Camada 2 : 32 neurônios (ReLU)')
print('  Camada 3 : 16 neurônios (ReLU)')
print('  Saída    : 1 neurônio (Sigmoid)')
print('  Otimizador: Adam | LR: 0.001')
print('\nTreinando...')

inicio = time.time()
modelo.fit(X_train_sc, y_train)
tempo = time.time() - inicio

print(f'\n✅ Treinamento concluído!')
print(f'   Épocas executadas : {modelo.n_iter_}')
print(f'   Loss final        : {modelo.loss_:.4f}')
print(f'   Tempo de treino   : {tempo:.2f}s')

Arquitetura da Rede Neural:
  Entrada  : 8 features
  Camada 1 : 64 neurônios (ReLU)
  Camada 2 : 32 neurônios (ReLU)
  Camada 3 : 16 neurônios (ReLU)
  Saída    : 1 neurônio (Sigmoid)
  Otimizador: Adam | LR: 0.001

Treinando...

✅ Treinamento concluído!
   Épocas executadas : 51
   Loss final        : 0.3870
   Tempo de treino   : 0.25s


## 📊 Célula 6 — Avaliação do Modelo
Avaliamos o modelo no conjunto de **teste** (dados que o modelo nunca viu).

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score)

y_pred      = modelo.predict(X_test_sc)
y_pred_prob = modelo.predict_proba(X_test_sc)[:, 1]

acuracia = accuracy_score(y_test, y_pred)
auc      = roc_auc_score(y_test, y_pred_prob)
cm       = confusion_matrix(y_test, y_pred)

print('=' * 50)
print('  MÉTRICAS DE AVALIAÇÃO')
print('=' * 50)
print(f'  Acurácia  : {acuracia:.4f} ({acuracia*100:.1f}%)')
print(f'  AUC-ROC   : {auc:.4f}')
print()
print('  Matriz de Confusão:')
print(f'  ┌─────────────────────────────┐')
print(f'  │           Previsto          │')
print(f'  │     Não-Diab  Com-Diab      │')
print(f'  │ Real Não: {cm[0][0]:>4}   {cm[0][1]:>4}         │')
print(f'  │ Real Com: {cm[1][0]:>4}   {cm[1][1]:>4}         │')
print(f'  └─────────────────────────────┘')
print()
print('  Relatório Completo:')
print(classification_report(y_test, y_pred,
      target_names=['Sem Diabetes', 'Com Diabetes']))

  MÉTRICAS DE AVALIAÇÃO
  Acurácia  : 0.7208 (72.1%)
  AUC-ROC   : 0.8154

  Matriz de Confusão:
  ┌─────────────────────────────┐
  │           Previsto          │
  │     Não-Diab  Com-Diab      │
  │ Real Não:   80     20         │
  │ Real Com:   23     31         │
  └─────────────────────────────┘

  Relatório Completo:
              precision    recall  f1-score   support

Sem Diabetes       0.78      0.80      0.79       100
Com Diabetes       0.61      0.57      0.59        54

    accuracy                           0.72       154
   macro avg       0.69      0.69      0.69       154
weighted avg       0.72      0.72      0.72       154



## 💾 Célula 7 — Salvar o Modelo
Salvamos **modelo + scaler** juntos em um único arquivo `.joblib`.
O scaler é essencial: sem ele, os dados novos não serão normalizados corretamente.

In [ ]:
import joblib
from google.colab import files

# Salvar modelo e scaler juntos
pacote = {
    'model'        : modelo,
    'scaler'       : scaler,
    'feature_names': feature_names,
    'acuracia'     : acuracia,
    'auc'          : auc,
    'n_iter'       : modelo.n_iter_
}

NOME_ARQUIVO = 'modelo_pima_mlp.joblib'
joblib.dump(pacote, NOME_ARQUIVO)

print(f'✅ Modelo salvo: {NOME_ARQUIVO}')
print(f'   Acurácia salva  : {acuracia*100:.1f}%')
print(f'   AUC-ROC salvo   : {auc:.4f}')
print(f'   Features salvas : {feature_names}')
print()
print('Fazendo download do arquivo...')
files.download(NOME_ARQUIVO)
print('✅ Download iniciado! Salve o arquivo para usar no Notebook 2.')

✅ Modelo salvo: modelo_pima_mlp.joblib
   Acurácia salva  : 72.1%
   AUC-ROC salvo   : 0.8154
   Features salvas : ['Gestacoes', 'Glicose', 'PressaoArterial', 'EspessuraPele', 'Insulina', 'IMC', 'LinhagemuDiabetes', 'Idade']

Fazendo download do arquivo...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download iniciado! Salve o arquivo para usar no Notebook 2.
